In [1]:
# conda activate genomic_tools

import os
import sys
import json
import pickle
import pandas as pd
from collections import defaultdict

sys.path.append("code")

from modified_functions import *

## Load GTF

In [2]:
# Load GTF

exclude = ""
gene_name = "gene_name"
gene_type = "all"
no_trim_id = False
gene_type_tag = "gene_type"
transcript_type_tag = "transcript_type"

gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v46.annotation.gtf"
gtf = process_gtf(gtf_file, exclude, gene_name, no_trim_id, gene_type_tag, transcript_type_tag)
gtf_cds = gtf[gtf.feature == "CDS"]
cds_by_transcript = {t: grp for t, grp in gtf_cds.groupby('transcript')}  # for transcript coding sequences lookup

Processing GTF file...


INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'level', 'tag', 'transcript_id', 'transcript_type', 'transcript_name', 'transcript_support_level', 'havana_transcript', 'exon_number', 'exon_id', 'hgnc_id', 'havana_gene', 'ont', 'protein_id', 'ccdsid', 'artif_dupl']


## Load interproscan results

In [3]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interproscan_results = pd.read_csv(
    "data/interproscan/results/proteins.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

In [4]:
# Parse the PIRSR data

with open("data/interproscan/interpro/data/pirsr/sr_uru.json") as f:
    pirsr_data = json.load(f)

# subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [5]:
interproscan_results = interproscan_results.merge(pirsr_df, left_on="signature_accession", right_on="accession", how="left")

## Map events to interproscan results

(Get the transcript associated with each significant event)

In [10]:
def get_skip_junction_aa(skip_transcript, exon_cds_start, exon_cds_end, cds_by_transcript):
    cds = _cds_rows(cds_by_transcript.get(skip_transcript))
    if cds is None:
        return None
    nt_before = 0
    for c in cds:
        if c['start'] == exon_cds_start and c['end'] == exon_cds_end:
            break   # reached the cassette exon row, stop
        nt_before += c['end'] - c['start'] + 1
    return nt_before // 3

def near_junction(df, junction_aa, window=50):
    """Keep features within window aa of junction_aa on either side."""
    return df[
        (df['stop'] >= junction_aa - window) &
        (df['start'] <= junction_aa + window)
    ]
    
def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]

In [7]:
# map events to interproscan results

with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

In [ ]:
columns = ['protein_accession', 'sequence_length', 'analysis', 
           'signature_description', 'start', 'stop', 'interpro_description']
analyses_to_exclude = ['NCBIFAM', 'SFLD']
ipr = interproscan_results.loc[~interproscan_results['analysis'].isin(analyses_to_exclude), columns]

ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

event_interproscan_map = defaultdict(dict)

for ev, rec in event_protein_map.items():

    rec_incl = rec['inclusion']
    incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
    if incl_df is None:
        continue
    overlap_df = incl_df[
        (incl_df['start'] <= rec_incl['aa_end']) &
        (incl_df['stop']  >= rec_incl['aa_start'])
    ]
    if overlap_df.empty:
        continue
    event_interproscan_map[ev]['inclusion'] = overlap_df.assign(
        aa_start = rec_incl['aa_start'],
        aa_end = rec_incl['aa_end'],
        exon_cds_start = rec_incl['exon_cds_start'],
        exon_cds_end = rec_incl['exon_cds_end'],
        frame_preserving = rec_incl['frame_preserving'],
        clean_start = rec_incl['clean_start'],
        clean_end = rec_incl['clean_end'],
    ).reset_index(drop=True)

    # real skip
    if rec.get('real_skip'):
        skip_df = ipr_grouped.get(rec['real_skip'])
        if skip_df is not None:
            junction_aa = get_skip_junction_aa(
                rec['real_skip'], rec_incl['exon_cds_start'],
                rec_incl['exon_cds_end'], cds_by_transcript
            ) or rec_incl['aa_start']
            s = near_junction(skip_df, junction_aa).assign(
                truncation_aa = junction_aa,
                frame_preserving = rec_incl['frame_preserving'],
            )
            if not s.empty:
                event_interproscan_map[ev]['real_skip'] = s.reset_index(drop=True)

    # synthetic skip
    if rec.get('synthetic_skip'):
        synth_df = ipr_grouped.get(rec['synthetic_skip'])
        if synth_df is not None:
            junction_aa = rec_incl['aa_start']
            s = near_junction(synth_df, junction_aa).assign(
                truncation_aa = junction_aa,
                frame_preserving = rec_incl['frame_preserving'],
            )
            if not s.empty:
                event_interproscan_map[ev]['synthetic_skip'] = s.reset_index(drop=True)

    # junction siblings
    if rec.get('exon_diff_junction_siblings'):
        frames = []
        for sib in rec['exon_diff_junction_siblings']:
            t = sib['transcript_id']
            if t not in ipr_grouped:
                continue
            sib_df = ipr_grouped[t]
            sib_overlap = sib_df[
                (sib_df['start'] <= sib['aa_end']) &
                (sib_df['stop']  >= sib['aa_start'])
            ].assign(
                aa_start = sib['aa_start'],
                aa_end = sib['aa_end'],
                exon_cds_start = sib['exon_cds_start'],
                exon_cds_end = sib['exon_cds_end'],
            )
            if not sib_overlap.empty:
                frames.append(sib_overlap)
        if frames:
            event_interproscan_map[ev]['exon_diff_junction_siblings'] = pd.concat(
                frames, ignore_index=True
            )

    # boundary siblings
    if rec.get('exon_diff_boundary_siblings'):
        frames = []
        for sib in rec['exon_diff_boundary_siblings']:
            t = sib['transcript_id']
            if t not in ipr_grouped:
                continue
            sib_df = ipr_grouped[t]
            sib_overlap = sib_df[
                (sib_df['start'] <= sib['aa_end']) &
                (sib_df['stop']  >= sib['aa_start'])
            ].assign(
                aa_start = sib['aa_start'],
                aa_end = sib['aa_end'],
                exon_cds_start = sib['exon_cds_start'],
                exon_cds_end = sib['exon_cds_end'],
            )
            if not sib_overlap.empty:
                frames.append(sib_overlap)
        if frames:
            event_interproscan_map[ev]['exon_diff_boundary_siblings'] = pd.concat(
                frames, ignore_index=True
            )

In [16]:
with open('data/event_interproscan_map.pkl', "wb") as file:
    pickle.dump(event_interproscan_map, file)

## Merge interproscan results with significant splicing event info.

In [17]:
# merge cell type-specific events with InterProScan results

signif_event_interproscan_map = dict() 
signif_event_interproscan_summary = dict()

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ctype = file.split("_exons.csv")[0]
        print(ctype)
        
        signif_events_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        event_interproscan_dict = {
            ev: event_interproscan_map[ev] for ev in signif_events_df.index 
            if ev in event_interproscan_map
        }

        result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
             for ev, buckets in event_interproscan_dict.items()
             for bucket, df in buckets.items()],
            ignore_index=True
        )
        # move event and bucket to front
        cols = ['event_id', 'bucket'] + [c for c in result.columns if c not in ('event_id', 'bucket')]
        result = result[cols]
 
        # restrict to splicing events with InterProScan results
        df = result.merge(signif_events_df, left_on="event_id", right_index=True)
        signif_event_interproscan_map[ctype] = df
        
        # summarize interpro results for significant splicing events
        signif_event_interproscan_summary[ctype] = df.groupby(["event_id", "bucket"]).agg(
            r=('r', lambda x: ' | '.join(map(str, x.unique()))),
            is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
            Gene=('Gene', lambda x: ' | '.join(x.unique())),
            interproscan_id=('protein_accession', lambda x: ' | '.join(x.unique())),
            chr=('chr', lambda x: ' | '.join(x.unique())),
            exon_start=('exon_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_end=('exon_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_len=('exon_len', lambda x: ' | '.join(map(str, x.unique()))),
            exon_cds_start=('exon_cds_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_cds_end=('exon_cds_end', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_start=('aa_start', lambda x: ' | '.join(map(str, x.unique()))),
            exon_aa_end=('aa_end', lambda x: ' | '.join(map(str, x.unique()))),
            domain_start=('start', lambda x: ' | '.join(map(str, x.unique()))),
            domain_stop=('stop', lambda x: ' | '.join(map(str, x.unique()))),
            protein_sequence_length=('sequence_length', lambda x: ' | '.join(map(str, x.unique()))),
            frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
            n_analyses=('analysis', lambda x: len(x.unique())),
            analyses=('analysis', lambda x: ' | '.join(x.unique())),
            signature_descriptions=('signature_description', lambda x: ' | '.join(x.unique())),
            interpro_descriptions=('interpro_description', lambda x: ' | '.join(x.unique()))
        ).reset_index()

Oligo
VLMC
Endo
Deep_layer_glutamatergic
Astro
OPC
Micro_PVM
All_Neuronal
All_GABAergic
Peri
CGE_Class
Upper_layer_glutamatergic


In [19]:
with open("data/signif_event_interproscan_map.pkl", "wb") as file:
    pickle.dump(signif_event_interproscan_map, file)
    
with open("data/signif_event_interproscan_summary.pkl", "wb") as file:
    pickle.dump(signif_event_interproscan_summary, file)

Preview

In [20]:
rec = signif_event_interproscan_summary['All_GABAergic']

rec[(rec['is_specific'] == "True") & (rec['bucket'] == "inclusion")].sort_values('n_analyses', ascending=False).head(10)

,event_id,bucket,r,is_specific,Gene,interproscan_id,chr,exon_start,exon_end,exon_len,...,exon_aa_start,exon_aa_end,domain_start,domain_stop,protein_sequence_length,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
667,ENSG00000049323_ProteinCoding_2,inclusion,-0.2017625375727214,True,LTBP1,ENST00000404816,chr2,33342838,33342963,126,...,1243.0,1285.0,1203 | 1244 | 1202 | 872 | 1201 | 1285 | 1224 ...,1243 | 1286 | 1328 | 1276 | 1314 | 1284 | 1316...,1721,True,10,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,Laminin | latent-transforming growth factor be...,- | Complement Clr-like EGF domain | EGF-like ...
2210,ENSG00000092096_ProteinCoding_2,inclusion,0.2085403902319068,True,SLC22A17,ENST00000354772,chr14,23351752,23351855,104,...,200.0,234.0,177 | 150 | 174 | 217 | 207 | 230 | 1 | 215 | ...,599 | 587 | 597 | 229 | 240 | 206 | 596 | 245 ...,631,False,10,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,MFS general substrate transporter like domains...,MFS transporter superfamily | - | Major facili...
4424,ENSG00000116983_ProteinCoding_1,inclusion,-0.2819875540122913,True,HPCAL4,ENST00000372844,chr1,39683937,39684152,216,...,54.0,125.0,1 | 65 | 100 | 17 | 66 | 98 | 64 | 73 | 109 | ...,189 | 190 | 125 | 176 | 86 | 90 | 175 | 92 | 1...,191,True,9,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,EF-hand | Visinin-like protein 1 | - | EF hand...,- | EF-hand domain | EF-hand domain pair | EF-...
5411,ENSG00000128342_ProteinCoding_1,inclusion,-0.239586841772764,True,LIF,ENST00000249075,chr22,30244755,30244933,179,...,6.0,65.0,23 | 43 | 3 | 33 | 15 | 1 | 34 | 24,202 | 14 | 32 | 191 | 22,202,False,9,CATH-Gene3D | CATH-FunFam | Pfam | Phobius | S...,- | leukemia inhibitory factor | LIF / OSM fam...,"Four-helical cytokine-like, core | - | Leukemi..."
7637,ENSG00000144355_ProteinCoding_1,inclusion,-0.2422743850073355,True,DLX1,ENST00000361725,chr2,172086654,172086853,200,...,104.0,170.0,123 | 129 | 95 | 100 | 128 | 125 | 161 | 126,189 | 185 | 118 | 112 | 190 | 184 | 186,255,False,9,CATH-Gene3D | CATH-FunFam | CDD | MobiDB-lite ...,Homeodomain-like | Distal-less homeobox 1 | - ...,- | Homeodomain | Homeodomain-like superfamily...
11257,ENSG00000179520_ProteinCoding_1,inclusion,0.3529908454564064,True,SLC17A8,ENST00000323346,chr12,100402596,100402745,150,...,301.0,350.0,310 | 79 | 117 | 271 | 334 | 345 | 73 | 75 | 315,508 | 502 | 463 | 309 | 344 | 333 | 370 | 503 ...,589,True,9,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,MFS general substrate transporter like domains...,MFS transporter superfamily | - | Major facili...
11732,ENSG00000185386_ProteinCoding_1,inclusion,0.4224290353048264,True,MAPK11,ENST00000330651,chr22,50267569,50267627,59,...,82.0,101.0,20 | 5 | 43 | 23 | 70 | 27 | 30 | 25 | 26 | 29...,346 | 145 | 218 | 239 | 225 | 258 | 216 | 176 ...,364,False,8,CATH-Gene3D | CATH-FunFam | PIRSR | Pfam | SMA...,Phosphorylase Kinase; domain 1 | Mitogen-activ...,- | Protein kinase domain | Protein kinase-lik...
323,ENSG00000012504_ProteinCoding_2,inclusion,0.2267839559121001,True,NR1H4,ENST00000548884,chr12,100532458,100532598,141,...,148.0,195.0,118 | 124 | 126 | 125 | 127,241 | 225 | 207 | 192 | 195 | 209 | 153 | 199,472,True,8,CATH-Gene3D | CATH-FunFam | CDD | Pfam | SMART...,"Erythroid Transcription Factor GATA-1, subunit...","Zinc finger, NHR/GATA-type | - | Zinc finger, ..."
3005,ENSG00000103546_ProteinCoding_2,inclusion,0.2164239724074509,True,SLC6A2,ENST00000568943,chr16,55696225,55696337,113,...,382.0,419.0,56 | 54 | 389 | 370 | 416 | 55 | 402,615 | 585 | 579 | 415 | 388 | 443 | 577 | 582 ...,617,False,8,CDD | PIRSR | Pfam | Phobius | SUPERFAMILY | P...,Na(+)- and Cl(-)-dependent norepinephrine tran...,- | Sodium:neurotransmitter symporter | Sodium...
5463,ENSG00000128655_ProteinCoding_1,inclusion,0.2241809700547417,True,PDE11A,ENST00000286063,chr2,177669493,177669567,75,...,829.0,853.0,584 | 585 | 663 | 607 | 661 | 608 | 588,911 | 839 | 917 | 897 | 830 | 914 | 912,933,True,8,CATH-Gene3D | CA

In [21]:
rec

,event_id,bucket,r,is_specific,Gene,interproscan_id,chr,exon_start,exon_end,exon_len,...,exon_aa_start,exon_aa_end,domain_start,domain_stop,protein_sequence_length,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
0,ENSG00000000419_ProteinCoding_2,exon_diff_boundary_siblings,-0.2452003297903593,False,DPM1,ENST00000371582,chr20,50941105,50941209,105,...,164.0,191.0,3 | 19 | 29 | 28 | 10,280 | 169 | 283 | 225 | 270,287,nan,5,CATH-Gene3D | CATH-FunFam | CDD | Pfam | SUPER...,Spore Coat Polysaccharide Biosynthesis Protein...,Nucleotide-diphospho-sugar transferases | - | ...
1,ENSG00000000419_ProteinCoding_2,inclusion,-0.2452003297903593,False,DPM1,ENST00000371584,chr20,50941105,50941209,105,...,164.0,199.0,3 | 19 | 29 | 28 | 10,284 | 169 | 291 | 233 | 278,295,True,5,CATH-Gene3D | CATH-FunFam | CDD | Pfam | SUPER...,Spore Coat Polysaccharide Biosynthesis Protein...,Nucleotide-diphospho-sugar transferases | - | ...
2,ENSG00000000419_ProteinCoding_2,real_skip,-0.2452003297903593,False,DPM1,ENST00000371588,chr20,50941105,50941209,105,...,nan,nan,3 | 19 | 29 | 28 | 10,253 | 255 | 256 | 198 | 243,260,nan,5,CATH-Gene3D | CATH-FunFam | CDD | Pfam | SUPER...,Spore Coat Polysaccharide Biosynthesis Protein...,Nucleotide-diphospho-sugar transferases | - | ...
3,ENSG00000000419_ProteinCoding_2,synthetic_skip,-0.2452003297903593,False,DPM1,ENSG00000000419_ProteinCoding_2_synthetic_skip,chr20,50941105,50941209,105,...,nan,nan,3 | 19 | 29 | 28 | 10,253 | 255 | 256 | 198 | 243,260,nan,5,CATH-Gene3D | CATH-FunFam | CDD | Pfam | SUPER...,Spore Coat Polysaccharide Biosynthesis Protein...,Nucleotide-diphospho-sugar transferases | - | ...
4,ENSG00000000419_ProteinCoding_3,inclusion,-0.2577583811672915,False,DPM1,ENST00000371582,chr20,50941129,50941209,81,...,164.0,191.0,3 | 19 | 29 | 28 | 10,280 | 169 | 283 | 225 | 270,287,True,5,CATH-Gene3D | CATH-FunFam | CDD | Pfam | SUPER...,Spore Coat Polysaccharide Biosynthesis Protein...,Nucleotide-diphospho-sugar transferases | - | ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13226,ENSG00000285043_ProteinCoding_3,inclusion,-0.2810183263051662,False,ENSG00000285043,ENST00000568435,chr16,30055172,30055327,156,...,2.0,53.0,1,22,91,True,1,MobiDB-lite,Consensus disorder prediction,-
13227,ENSG00000285043_ProteinCoding_3,real_skip,-0.2810183263051662,False,ENSG00000285043,ENST00000562240,chr16,30055172,30055327,156,...,nan,nan,20 | 1 | 46 | 24 | 26,45 | 19 | 87 | 46,87,nan,3,Phobius | TMbed | DeepTMHMM,Transmembrane region | Non cytoplasmic domain ...,-
13228,ENSG00000288701_ProteinCoding_2,inclusion,0.4401034072379004,False,PRRC2B,ENST00000683519,chr9,131474454,131476535,2082,...,774.0,1468.0,949 | 830 | 897 | 950 | 1016 | 1107 | 1111 | 1...,979 | 1045 | 933 | 1039 | 1528 | 1123 | 1195 |...,2229,True,2,COILS | MobiDB-lite,Coil | Consensus disorder prediction,-
13229,ENSG00000288701_ProteinCoding_2,real_skip,0.4401034072379004,False,PRRC2B,ENST00000682501,chr9,131474454,131476535,2082,...,nan,nan,496 | 869 | 49 | 88 | 115 | 219 | 386 | 423 | ...,552 | 889 | 269 | 107 | 137 | 248 | 637 | 436 ...,1535,nan,3,COILS | MobiDB-lite | Pfam,Coil | Consensus disorder prediction | BAT2 N-...,"- | BAT2, N-terminal"
